# UdaPlay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Optional SQLite compatibility for the Udacity workspace.
# pysqlite3 is imported dynamically so local editors do not warn when it is absent.

from importlib import import_module, util
import sys

if util.find_spec("pysqlite3") is not None:
    pysqlite3 = import_module("pysqlite3")
    sys.modules["sqlite3"] = pysqlite3

In [ ]:
import os

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.evaluation import AgentEvaluator, TestCase
from lib.llm import LLM
from lib.messages import AIMessage, SystemMessage, ToolMessage, UserMessage
from lib.parsers import PydanticOutputParser
from lib.tooling import tool
from lib.vector_db import VectorStore


In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

assert OPENAI_API_KEY is not None, "OPENAI_API_KEY must be set in your local .env file"
assert TAVILY_API_KEY is not None, "TAVILY_API_KEY must be set in your local .env file"

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
# Connect to the collection once, reusing the same embedding function as Part 01
# so query text is embedded into the same vector space as the stored games.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
)
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)


@tool
def retrieve_game(query: str):
    """Semantic search: Finds most similar results in the vector DB

    args:
    - query: a question about game industry. 
    """

    # Embed the query and find the nearest stored games
    results = collection.query(
        query_texts=[query],
        n_results=5
    )

    # Chroma returns a list per query; we only sent one query
    metadatas = results["metadatas"][0]
    documents = results["documents"][0]

    # Build a clean list to return
    formatted_results = []
    for metadata, document in zip(metadatas, documents):
        formatted_results.append({
            "Platform": metadata.get("Platform", "Unknown"),
            "Name": metadata.get("Name", "Unknown"),
            "YearOfRelease": metadata.get("YearOfRelease", "Unknown"),
            "Description": document
        })

    return formatted_results

#### Evaluate Retrieval Tool

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(
        description="Whether the retrieved documents are enough to answer the question"
    )
    description: str = Field(
        description="Detailed explanation of the evaluation result"
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> EvaluationReport:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    """

    # Ask an LLM to judge whether the documents can answer the question
    llm = LLM(model="gpt-4o-mini", temperature=0.0)

    system_message = SystemMessage(
        content=(
            "Your task is to evaluate if the documents are enough to respond the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not."
        )
    )
    user_message = UserMessage(
        content=f"Question: {question}\nRetrieved documents: {retrieved_docs}"
    )

    # Request a structured response and parse it into EvaluationReport
    ai_message = llm.invoke(
        input=[system_message, user_message],
        response_format=EvaluationReport,
    )
    parser = PydanticOutputParser(model_class=EvaluationReport)
    return parser.parse(ai_message)

In [ ]:
# Quick check: a relevant query should be useful, an off-topic one should not.
relevant_question = "What is Mario Kart?"
relevant_docs = retrieve_game(relevant_question)
relevant_report = evaluate_retrieval(relevant_question, relevant_docs)
print("useful:", relevant_report.useful)
print("description:", relevant_report.description)

print("-" * 40)

offtopic_question = "What is the best recipe for pizza?"
offtopic_docs = retrieve_game(offtopic_question)
offtopic_report = evaluate_retrieval(offtopic_question, offtopic_docs)
print("useful:", offtopic_report.useful)
print("description:", offtopic_report.description)

#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

### Agent

In [ ]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes